In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# Load dataset
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Example: assume target column is 'target'
X = train.drop("target", axis=1)
y = train["target"]

# Train-test split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
}

# Train and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, preds)
    print(f"{name} MAE: {mae:.4f}")

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
import xgboost as xgb


In [17]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [18]:
cat_cols = ["Sex"]
num_cols = [c for c in train.columns if c not in ["Age", "Sex"]]

X = train.drop("Age", axis=1)
y = train["Age"]

In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
        ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
    ],
    remainder="drop",
)


In [21]:
stacking_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            StackingRegressor(
                estimators=[
                    (
                        "rf",
                        RandomForestRegressor(
                            n_estimators=400,
                            max_depth=12,
                            random_state=42,
                            n_jobs=-1,
                        ),
                    ),
                    (
                        "gbr",
                        GradientBoostingRegressor(
                            n_estimators=500,
                            learning_rate=0.05,
                            max_depth=5,
                            random_state=42,
                        ),
                    ),
                ],
                final_estimator=Ridge(alpha=0.5),
                cv=5,
                n_jobs=-1,
                passthrough=False,
            ),
        ),
    ]
)

models = {
    "Stacking_RF_GB": stacking_model,
}


In [23]:
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, preds)
    print(f"{name} MAE: {mae:.4f}")

Stacking_RF_GB MAE: 1.3806
